In [1]:
import sys, pandas, sklearn
print(sys.executable)

C:\Users\dmiva\.conda\envs\anaconda-mcp\envs\progetto_icon\python.exe


In [2]:
import pandas as pd

path = "../data/games.csv"

cols = pd.read_csv(path, nrows=0).columns.tolist()
if "DiscountDLC count" in cols:
    i = cols.index("DiscountDLC count")
    cols[i:i+1] = ["Discount", "DLC count"]

df = pd.read_csv(path, header=None, skiprows=1, names=cols, index_col=False)

In [3]:
df[["AppID", "Name", "Release date", "Price", "Positive", "Negative"]].head()

,AppID,Name,Release date,Price,Positive,Negative
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0.00,0,0
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",5.24,252,3
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",4.99,21,3
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",8.99,0,0
4,3631080,Maze Quest VR,"Apr 24, 2025",4.99,0,0


In [4]:
df.shape

(125855, 40)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125855 non-null  int64  
 1   Name                        125854 non-null  str    
 2   Release date                125855 non-null  str    
 3   Estimated owners            125855 non-null  str    
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  int64  
 6   Price                       125855 non-null  float64
 7   Discount                    125855 non-null  int64  
 8   DLC count                   125855 non-null  int64  
 9   About the game              117393 non-null  str    
 10  Supported languages         125855 non-null  str    
 11  Full audio languages        125855 non-null  str    
 12  Reviews                     12181 non-null   str    
 13  Header image             

In [6]:
df.isna().sum().sort_values(ascending=False).head(15)

Movies            125855
Score rank        125815
Metacritic url    121597
Reviews           113674
Notes             102545
Website            75335
Support url        70560
Tags               42502
Support email      22630
Categories          8966
Publishers          8922
About the game      8462
Developers          8449
Genres              8423
Screenshots         6027
dtype: int64

In [7]:
df[["Price", "Positive", "Negative", "Peak CCU", "Metacritic score"]].describe()

,Price,Positive,Negative,Peak CCU,Metacritic score
count,125855.000000,1.258550e+05,1.258550e+05,1.258550e+05,125855.000000
mean,4.811174,1.018182e+03,1.648786e+02,5.318720e+01,2.500083
std,12.489329,2.772782e+04,5.305001e+03,3.681083e+03,13.492922
min,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,0.590000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
50%,2.390000,4.000000e+00,1.000000e+00,0.000000e+00,0.000000
75%,5.545000,3.500000e+01,1.000000e+01,0.000000e+00,0.000000
max,999.980000,7.642084e+06,1.173003e+06,1.013936e+06,97.000000


In [8]:
df["n_recensioni"] = df["Positive"] + df["Negative"]
print((df["n_recensioni"] == 0).sum(), "giochi con 0 recensioni")
print(df["n_recensioni"].describe())

42899 giochi con 0 recensioni
count    1.258550e+05
mean     1.183061e+03
std      3.221401e+04
min      0.000000e+00
25%      0.000000e+00
50%      6.000000e+00
75%      4.600000e+01
max      8.815087e+06
Name: n_recensioni, dtype: float64


In [9]:
df.to_pickle("../data/games.pkl")

In [10]:
soglie = [1, 5, 10, 20, 50, 100]
for s in soglie:
    n = (df["n_recensioni"] >= s).sum()
    print(f"almeno {s:>3} recensioni: {n:>6} giochi ({n/len(df):.1%})")

almeno   1 recensioni:  82956 giochi (65.9%)
almeno   5 recensioni:  67142 giochi (53.3%)
almeno  10 recensioni:  56662 giochi (45.0%)
almeno  20 recensioni:  44568 giochi (35.4%)
almeno  50 recensioni:  30624 giochi (24.3%)
almeno 100 recensioni:  22804 giochi (18.1%)


In [11]:
df["perc_positive"] = df["Positive"] / df["n_recensioni"]
df.loc[df["n_recensioni"] >= 10, "perc_positive"].describe()

count    56662.000000
mean         0.772225
std          0.178922
min          0.000000
25%          0.672727
50%          0.812500
75%          0.910172
max          1.000000
Name: perc_positive, dtype: float64

In [12]:
SOGLIA_RECENSIONI = 10
SOGLIA_APPREZZATO = 0.80

dati = df[df["n_recensioni"] >= SOGLIA_RECENSIONI].copy()
dati["apprezzato"] = (dati["perc_positive"] >= SOGLIA_APPREZZATO).astype(int)

print(dati.shape)
print(dati["apprezzato"].value_counts())
print(dati["apprezzato"].value_counts(normalize=True))

(56662, 43)
apprezzato
1    30313
0    26349
Name: count, dtype: int64
apprezzato
1    0.534979
0    0.465021
Name: proportion, dtype: float64


In [13]:
dati.to_pickle("../data/games_target.pkl")

In [14]:
colonne_feature = [
    "Release date", "Price", "Required age",
    "Windows", "Mac", "Linux",
    "Supported languages", "Full audio languages",
    "Developers", "Publishers", "Genres", "Categories",
    "Achievements", "Screenshots", "About the game",
    "Website", "Support url", "Support email",
]
target = "apprezzato"

dataset = dati[colonne_feature + [target]].copy()
print(dataset.shape)
dataset.head()

(56662, 19)


,Release date,Price,Required age,Windows,Mac,Linux,Supported languages,Full audio languages,Developers,Publishers,Genres,Categories,Achievements,Screenshots,About the game,Website,Support url,Support email,apprezzato
1,"Jul 29, 2016",5.24,0,True,False,False,['English'],[],minori,MangaGamer,Adventure,"Single-player,Steam Trading Cards,Steam Cloud,...",0,https://shared.akamai.steamstatic.com/store_it...,"Springtime, April: when the cherry trees come ...",http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,1
2,"May 6, 2019",4.99,0,True,True,False,"['English', 'French', 'German', 'Russian']",[],Somer Games,8floor,Casual,"Single-player,Family Sharing",0,https://shared.akamai.steamstatic.com/store_it...,"Immerse yourself in the most beloved, mystical...",https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,1
6,"Apr 8, 2025",35.99,0,True,False,False,['English'],[],Veitikka Studios,Matrix Games,"Simulation,Strategy","Single-player,Steam Workshop,Steam Cloud,Famil...",0,https://shared.akamai.steamstatic.com/store_it...,Building on the success of its acclaimed prede...,NaN,NaN,support@slitherine.co.uk,1
8,"May 7, 2021",0.99,0,True,False,False,"['English', 'French', 'Italian', 'German', 'Sp...","['English', 'French', 'Italian', 'German', 'Sp...",Matheus Bigatão,Matheus Bigatão,"Casual,Indie","Single-player,Steam Achievements,Full controll...",9,https://shared.akamai.steamstatic.com/store_it...,Hepta Beats is a 2D side-view platformer game ...,NaN,NaN,heptabeats@gmail.com,1
9,"Dec 13, 2021",0.59,0,True,True,True,"['English', 'French', 'Italian', 'German', 'Sp...",[],Airem,Airem,"Action,Casual,Indie,Strategy","Single-player,Steam Achievements,Full controll...",100,https://shared.akamai.steamstatic.com/store_it...,WARRNING Don't show Your mom this game. You kn...,http://airem.pl,http://airem.pl,info@airem.pl,0


In [15]:
feat = pd.DataFrame(index=dataset.index)

data = pd.to_datetime(dataset["Release date"], format="mixed", errors="coerce")
feat["anno"] = data.dt.year
feat["mese"] = data.dt.month
print(data.isna().sum(), "date non riconosciute")

0 date non riconosciute


In [16]:
feat["prezzo"] = dataset["Price"]
feat["eta_minima"] = dataset["Required age"]
feat["achievements"] = dataset["Achievements"]

for col in ["Windows", "Mac", "Linux"]:
    feat[col.lower()] = dataset[col].astype(int)

In [17]:
feat["ha_sito"] = dataset["Website"].notna().astype(int)
feat["ha_supporto_url"] = dataset["Support url"].notna().astype(int)
feat["ha_supporto_email"] = dataset["Support email"].notna().astype(int)

In [18]:
def conta_elementi(testo):
    if pd.isna(testo) or testo in ("", "[]"):
        return 0
    return testo.count(",") + 1

feat["n_lingue"] = dataset["Supported languages"].apply(conta_elementi)
feat["n_lingue_audio"] = dataset["Full audio languages"].apply(conta_elementi)
feat["n_screenshot"] = dataset["Screenshots"].apply(conta_elementi)
feat["lunghezza_descrizione"] = dataset["About the game"].fillna("").str.len()

In [19]:
print(feat.shape)
feat.describe().T

(56662, 15)


,count,mean,std,min,25%,50%,75%,max
anno,56662.0,2019.885902,3.491306,1997.0,2017.00,2020.00,2023.00,2025.00
mese,56662.0,6.547863,3.424373,1.0,4.00,7.00,10.00,12.00
prezzo,56662.0,5.212011,10.670397,0.0,0.99,2.99,6.29,299.99
eta_minima,56662.0,0.296336,2.188732,0.0,0.00,0.00,0.00,18.00
achievements,56662.0,29.016343,206.235540,0.0,0.00,11.00,27.00,9821.00
windows,56662.0,0.999894,0.010290,0.0,1.00,1.00,1.00,1.00
mac,56662.0,0.232943,0.422710,0.0,0.00,0.00,0.00,1.00
linux,56662.0,0.160460,0.367036,0.0,0.00,0.00,0.00,1.00
ha_sito,56662.0,0.545463,0.497933,0.0,0.00,1.00,1.00,1.00
ha_supporto_url,56662.0,0.539939,0.498407,0.0,0.00,1.00,1.00,1.00


In [20]:
confronto = feat.copy()
confronto["apprezzato"] = dataset["apprezzato"]

for col in ["ha_sito", "ha_supporto_url", "ha_supporto_email", "mac", "linux"]:
    perc = confronto.groupby(col)["apprezzato"].mean()
    print(f"{col:>18}: senza = {perc[0]:.1%}   con = {perc[1]:.1%}")

print()
fasce = pd.qcut(confronto["n_screenshot"], 4, duplicates="drop")
print(confronto.groupby(fasce, observed=True)["apprezzato"].mean())

           ha_sito: senza = 53.4%   con = 53.6%
   ha_supporto_url: senza = 54.1%   con = 53.0%
 ha_supporto_email: senza = 53.9%   con = 53.4%
               mac: senza = 51.3%   con = 60.8%
             linux: senza = 51.9%   con = 62.0%

n_screenshot
(-0.001, 6.0]    0.525954
(6.0, 8.0]       0.555831
(8.0, 11.0]      0.545928
(11.0, 183.0]    0.519967
Name: apprezzato, dtype: float64


In [21]:
da_rimuovere = ["windows", "ha_sito", "ha_supporto_url", "ha_supporto_email", "n_screenshot"]
feat = feat.drop(columns=da_rimuovere)
print(feat.columns.tolist())

['anno', 'mese', 'prezzo', 'eta_minima', 'achievements', 'mac', 'linux', 'n_lingue', 'n_lingue_audio', 'lunghezza_descrizione']


In [22]:
generi = dataset["Genres"].str.get_dummies(sep=",")
print(generi.shape)
generi.sum().sort_values(ascending=False)

(56662, 28)


Indie                    40583
Adventure                23961
Action                   23898
Casual                   21804
Simulation               12660
Strategy                 11613
RPG                      10951
Early Access              4673
Free To Play              4231
Sports                    2540
Racing                    2054
Massively Multiplayer     1404
Utilities                  526
Design & Illustration      336
Violent                    282
Animation & Modeling       275
Education                  207
Video Production           183
Gore                       182
Game Development           149
Audio Production           132
Software Training          111
Photo Editing               85
Nudity                      79
Web Publishing              72
Sexual Content              67
Accounting                   8
Movie                        1
dtype: int64

In [23]:
SOGLIA_GENERE = 0.01

frequenza = generi.mean()
generi_tenuti = frequenza[frequenza >= SOGLIA_GENERE].index.tolist()
print(len(generi_tenuti), generi_tenuti)

feat = feat.join(generi[generi_tenuti].add_prefix("genere_"))
print(feat.shape)

12 ['Action', 'Adventure', 'Casual', 'Early Access', 'Free To Play', 'Indie', 'Massively Multiplayer', 'RPG', 'Racing', 'Simulation', 'Sports', 'Strategy']
(56662, 22)


In [24]:
print(feat.isna().sum().sum(), "valori mancanti in feat")

finale = feat.copy()
finale["apprezzato"] = dataset["apprezzato"]
finale.to_pickle("../data/dataset_finale.pkl")
print(finale.shape)

0 valori mancanti in feat
(56662, 23)
